In [0]:
CATALOG_NAME = "workspace"
SCHEMA_NAME = "marketing_campaign"
VOLUME_NAME = "raw_files"

RAW_FILE_NAME = "marketing_campaign.csv"
RAW_FILE_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}/{RAW_FILE_NAME}"

BRONZE_TABLE_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_marketing_campaign"

print(f"Raw file path: {RAW_FILE_PATH}")
print(f"Bronze table: {BRONZE_TABLE_NAME}")

In [0]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", "\t")
    .csv(RAW_FILE_PATH)
)

display(raw_df.limit(10))

In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_df = (
    raw_df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit(RAW_FILE_PATH))
)

display(bronze_df.limit(10))

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE_NAME)
)

In [0]:
bronze_check_df = spark.table(BRONZE_TABLE_NAME)

print(f"Rows: {bronze_check_df.count()}")
print(f"Columns: {len(bronze_check_df.columns)}")

display(bronze_check_df.limit(10))

In [0]:
spark.sql(f"DESCRIBE TABLE {BRONZE_TABLE_NAME}").display()